# AIProjectClient를 활용한 기본 RAG (Retrieval-Augmented Generation)

이 노트북에서는 아래 SDK를 활용해 **기본 RAG 흐름**을 구현합니다.
- `azure-ai-projects` (`AIProjectClient`)
- `azure-ai-inference` (임베딩, Chat Completions)
- `azure-ai-search` (벡터/하이브리드 검색)

주제는 **건강/피트니스**입니다. 샘플 팁 문서를 만들고 임베딩한 뒤, 검색 인덱스에 저장하고, 사용자 질의에 맞는 문서를 검색해 최종 답변을 생성합니다.

> 안내: 이 노트북은 의료 조언을 제공하지 않습니다. 실제 건강 문제는 반드시 전문가와 상담하세요.

## RAG란?
RAG(Retrieval-Augmented Generation)는 LLM이 외부 데이터에서 검색한 관련 문서를 근거로 답변을 생성하는 방식입니다. 이를 통해 답변의 근거성을 높이고 환각(hallucination)을 줄일 수 있습니다.

## 1. 설정

라이브러리를 임포트하고 환경 변수를 불러옵니다.

- `PROJECT_ENDPOINT`/`PROJECT_API_KEY`: 프로젝트 클라이언트 및 Chat/Embedding 호출 공통
- `MODEL_NAME`/`TEXT_EMBEDDING_MODEL`: 배포 모델 이름
- `SEARCH_ENDPOINT`/`SEARCH_API_KEY`/`SEARCH_INDEX_NAME`: Azure AI Search 연결 정보

모델 호출용 endpoint는 코드에서 `PROJECT_ENDPOINT`를 기준으로 `/models`를 자동 구성합니다.

> 시작 전에 `2-embeddings.ipynb`를 먼저 실행해 임베딩 흐름을 익혀두면 좋습니다.

In [ ]:
# 이 노트북에서 필요한 패키지 설치 (커널당 1회)
%pip install -q python-dotenv azure-ai-projects azure-ai-inference azure-search-documents azure-core

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv

# 프로젝트 클라이언트
from azure.ai.projects import AIProjectClient
from azure.core.credentials import AzureKeyCredential

# 임베딩 및 채팅 모델
from azure.ai.inference import EmbeddingsClient, ChatCompletionsClient
from azure.ai.inference.models import UserMessage, SystemMessage

# 벡터/하이브리드 검색
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

def _clean_env(value: str | None) -> str | None:
    if value is None:
        return None
    return value.strip().strip('"').strip("'")

def _derive_models_endpoint(project_endpoint: str | None) -> str | None:
    if not project_endpoint:
        return None
    parsed = urlparse(project_endpoint)
    if not parsed.scheme or not parsed.netloc:
        return None
    return f"{parsed.scheme}://{parsed.netloc}/models"

dotenv_candidates = [
    Path.cwd() / '.env',
    Path.cwd().parent / '.env',
    Path('.env'),
    Path('..') / '.env',
]
dotenv_path = next((p for p in dotenv_candidates if p.exists()), None)
if dotenv_path:
    load_dotenv(dotenv_path, override=False)
    print(f"✅ .env 로드 완료: {dotenv_path}")
else:
    print("⚠️ .env 파일을 찾지 못했습니다. 현재/상위 폴더를 확인하세요.")

project_endpoint = _clean_env(os.environ.get("PROJECT_ENDPOINT"))
project_api_key = _clean_env(os.environ.get("PROJECT_API_KEY"))
endpoint = _derive_models_endpoint(project_endpoint)
api_key = project_api_key
chat_model = _clean_env(os.environ.get("MODEL_NAME"))
embedding_model = _clean_env(os.environ.get("TEXT_EMBEDDING_MODEL"))
search_index_name = _clean_env(os.environ.get("SEARCH_INDEX_NAME"))

if endpoint:
    print("ℹ️ Models endpoint 자동 구성 완료: <host>/models")

project_client = None
missing_keys = [
    name
    for name, value in {
        "PROJECT_ENDPOINT": project_endpoint,
        "PROJECT_API_KEY": project_api_key,
        "MODEL_NAME": chat_model,
        "TEXT_EMBEDDING_MODEL": embedding_model,
        "SEARCH_INDEX_NAME": search_index_name,
    }.items()
    if not value
]

if missing_keys:
    print(f"❌ 필수 환경 변수가 없습니다: {', '.join(missing_keys)}")
else:
    try:
        project_client = AIProjectClient(
            endpoint=project_endpoint,
            credential=AzureKeyCredential(project_api_key),
        )
        print("✅ AIProjectClient 생성 완료")
    except Exception as e:
        print("❌ AIProjectClient 생성 오류:", e)

## 2. 샘플 건강 데이터 생성

간단한 문서 조각을 생성합니다. 실제 환경에서는 CSV/PDF를 읽어 청크로 분할한 뒤 임베딩하고 인덱스에 저장합니다.

In [ ]:
health_tips = [
    {
        "id": "doc1",
        "content": "하루 30분 걷기는 체중 관리와 스트레스 완화에 도움이 됩니다.",
        "source": "기초 체력"
    },
    {
        "id": "doc2",
        "content": "하루에 물 8~10잔을 마시면 수분 균형 유지에 도움이 됩니다.",
        "source": "기초 체력"
    },
    {
        "id": "doc3",
        "content": "7~9시간의 규칙적인 수면은 근육 회복에 긍정적인 영향을 줍니다.",
        "source": "회복 관리"
    },
    {
        "id": "doc4",
        "content": "심폐 지구력을 높이려면 인터벌 트레이닝(HIIT)을 시도해 보세요.",
        "source": "운동 루틴"
    },
    {
        "id": "doc5",
        "content": "달리기 전에는 동적 스트레칭으로 워밍업해 부상 위험을 줄이세요.",
        "source": "운동 루틴"
    },
    {
        "id": "doc6",
        "content": "균형 잡힌 식단에는 단백질, 통곡물, 과일, 채소, 건강한 지방이 포함됩니다.",
        "source": "영양 관리"
    },
    {
        "id": "doc7",
        "content": "시간이 부족한 날에는 10분 유산소 루틴(제자리 걷기 2분, 버피 1분, 마운틴클라이머 1분, 스쿼트 2분, 가벼운 걷기 2분, 스트레칭 2분)을 권장합니다.",
        "source": "운동 루틴"
    },
]

print("✅ 샘플 건강 팁 문서 목록을 생성했습니다.")

## 3.0. 인덱스 생성 또는 초기화

Azure AI Search에서 벡터 필드를 사용할 때는 필드의 `vector_search_profile_name`이 벡터 검색 설정의 프로필 이름과 일치해야 합니다.

아래 셀에서 HNSW 기반 벡터 인덱스를 생성/초기화하는 헬퍼 함수를 정의합니다.

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchAlgorithmKind,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
)

def create_healthtips_index(
    endpoint: str,
    api_key: str,
    index_name: str,
    dimension: int = 1536,
):
    """건강 팁 문서용 벡터 검색 인덱스를 생성 또는 초기화합니다."""
    index_client = SearchIndexClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))

    # 기존 인덱스가 있으면 삭제
    try:
        index_client.delete_index(index_name)
        print(f"기존 인덱스를 삭제했습니다: {index_name}")
    except Exception:
        pass

    # 벡터 검색 설정
    vector_search = VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="myHnsw",
                kind=VectorSearchAlgorithmKind.HNSW,
                parameters=HnswParameters(
                    m=4,
                    ef_construction=400,
                    ef_search=500,
                    metric=VectorSearchAlgorithmMetric.COSINE,
                ),
            )
        ],
        profiles=[
            VectorSearchProfile(
                name="myHnswProfile",
                algorithm_configuration_name="myHnsw",
            )
        ],
    )

    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SimpleField(name="source", type=SearchFieldDataType.String),
        SearchField(
            name="embedding",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            vector_search_dimensions=dimension,
            vector_search_profile_name="myHnswProfile",
        ),
    ]

    index_def = SearchIndex(
        name=index_name,
        fields=fields,
        vector_search=vector_search,
    )

    index_client.create_index(index_def)
    print(f"✅ 인덱스를 생성(또는 초기화)했습니다: {index_name}")

## 3.1. 인덱스 생성 및 건강 팁 업로드

다음 순서로 지식 기반을 구성합니다.
1. 임베딩 클라이언트를 생성하고 벡터 길이를 확인합니다.
2. 벡터 검색 기능이 포함된 인덱스를 생성합니다.
3. 각 건강 팁에 대한 임베딩을 생성합니다.
4. 임베딩과 함께 문서를 인덱스에 업로드합니다.

In [ ]:
search_endpoint = _clean_env(os.environ.get("SEARCH_ENDPOINT"))
search_api_key = _clean_env(os.environ.get("SEARCH_API_KEY"))

# 1) 임베딩 클라이언트 생성 및 벡터 길이 확인
embeddings_client = EmbeddingsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(api_key),
    model=embedding_model,
)
print("✅ EmbeddingsClient 생성 완료")

sample_doc = health_tips[0]
emb_response = embeddings_client.embed(input=[sample_doc["content"]])
embedding_length = len(emb_response.data[0].embedding)
print(f"✅ 임베딩 길이 확인: {embedding_length}")

# 2) 인덱스 생성
create_healthtips_index(
    endpoint=search_endpoint,
    api_key=search_api_key,
    index_name=search_index_name,
    dimension=embedding_length,
)

# 3) 문서 업로드용 검색 클라이언트 생성
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=search_index_name,
    credential=AzureKeyCredential(search_api_key),
)
print("✅ SearchClient 생성 완료")

# 4) 문서 임베딩 생성 후 업로드
search_docs = []
for doc in health_tips:
    emb_response = embeddings_client.embed(input=[doc["content"]])
    emb_vec = emb_response.data[0].embedding
    search_docs.append({
        "id": doc["id"],
        "content": doc["content"],
        "source": doc["source"],
        "embedding": emb_vec,
    })

search_client.upload_documents(documents=search_docs)
print(f"✅ 총 {len(search_docs)}개 문서를 인덱스 '{search_index_name}'에 업로드했습니다.")

## 4. 기본 RAG 흐름

### 4.1 검색 (Retrieve)
1. 사용자 질문을 임베딩합니다.
2. 임베딩 벡터로 검색 인덱스에서 상위 문서를 찾습니다.

### 4.2 응답 생성 (Generate)
검색된 문서를 채팅 모델에 전달해 최종 답변을 생성합니다.

> 실제 운영에서는 청크 분할, 재랭킹, 요약 등 고도화 전략을 함께 사용합니다.

In [ ]:
from azure.search.documents.models import VectorizedQuery

def rag_chat(query: str, top_k: int = 5) -> str:
    # 1) 사용자 질문 임베딩
    user_vec = embeddings_client.embed(input=[query]).data[0].embedding

    # 2) 하이브리드 검색(텍스트 + 벡터)
    vector_query = VectorizedQuery(
        vector=user_vec,
        k_nearest_neighbors=top_k,
        fields="embedding",
    )

    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        select=["content", "source"],
        top=top_k,
    )

    top_docs_content = []
    for r in results:
        c = r["content"]
        s = r["source"]
        top_docs_content.append(f"출처: {s} | 내용: {c}")

    # 3) 검색 문서를 근거로 답변 생성
    system_text = (
        "당신은 건강/피트니스 어시스턴트입니다.\n"
        "아래 문서 내용만 근거로 답변하세요.\n"
        "문서 목록:\n"
        + "\n".join(top_docs_content)
        + "\n문서에 근거가 부족하면 '확실하지 않습니다'라고 답변하세요.\n"
    )

    chat_client = ChatCompletionsClient(
        endpoint=endpoint,
        credential=AzureKeyCredential(api_key),
        model=chat_model,
    )

    response = chat_client.complete(
        messages=[
            SystemMessage(content=system_text),
            UserMessage(content=query),
        ],
    )

    return response.choices[0].message.content

## 5. 질의 실행

바쁜 사람을 위한 짧은 유산소 운동 루틴을 질문해 결과를 확인해봅니다.

In [ ]:
user_query = "평일에 시간이 부족한 사람에게 적합한 짧은 유산소 루틴이 있을까요?"
answer = rag_chat(user_query)
print("🗣️ 사용자 질문:", user_query)
print("🤖 RAG 답변:", answer)

In [ ]:
# 진단: rag_chat과 동일 조건으로 검색 결과 확인
probe_query = "평일에 시간이 부족한 사람에게 적합한 짧은 유산소 루틴이 있을까요?"
probe_vec = embeddings_client.embed(input=[probe_query]).data[0].embedding

probe_results = search_client.search(
    search_text=probe_query,
    vector_queries=[VectorizedQuery(vector=probe_vec, k_nearest_neighbors=5, fields="embedding")],
    select=["id", "content", "source"],
    top=5,
)

rows = list(probe_results)
print("검색 결과 개수:", len(rows))
for i, r in enumerate(rows, 1):
    print(f"[{i}] source={r.get('source')} | content={r.get('content')}")

## 6. 마무리

이번 노트북에서는 다음을 포함한 기본 RAG 파이프라인을 시연했습니다.
- 문서를 임베딩해 Azure AI Search에 저장
- 사용자 질문과 관련된 상위 문서를 검색
- 검색 결과를 근거로 채팅 응답 생성

이 흐름은 고급 청크 분할, 하이브리드 검색, 품질 평가 단계를 추가해 확장할 수 있습니다.